# SENet — Squeeze-and-Excitation Networks (2017)

_Papers · Computer Vision_

**Let each channel look at the whole feature map, then turn itself up or down — a learned per-channel volume knob.**

---

This notebook is a practice scaffold for the **SENet — Squeeze-and-Excitation Networks (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torchvision, torchvision.transforms as T

torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked example (C=2, 2x2 maps, r=2). ---
u1 = torch.tensor([[4., 0.], [0., 0.]])
u2 = torch.tensor([[1., 1.], [1., 1.]])
z  = torch.stack([u1.mean(), u2.mean()])           # Eqn. 2 squeeze -> [1., 1.]
W1 = torch.tensor([[0.5, -0.5]])                   # 1x2  (C/r = 1 hidden unit)
W2 = torch.tensor([[2.0], [-2.0]])                 # 2x1
h  = torch.relu(W1 @ z)                             # FC1 + ReLU  -> 0.0
s  = torch.sigmoid(W2 @ h)                          # FC2 + sigmoid -> [0.5, 0.5]
print("worked example:  z =", z.tolist(), " gates s =", s.tolist())
# worked example:  z = [1.0, 1.0]  gates s = [0.5, 0.5]
print("scaled u1 =\n", (s[0] * u1).tolist(), "\nscaled u2 =\n", (s[1] * u2).tolist())
# scaled u1 = [[2.0, 0.0], [0.0, 0.0]]   scaled u2 = [[0.5, 0.5], [0.5, 0.5]]


# --- 1. The Squeeze-and-Excitation block, built by hand (Eqns. 2-4). ---
class SEBlock(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        hidden = max(1, channels // r)             # C/r bottleneck (>=1 for narrow stages)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        B, C, _, _ = x.shape
        z = x.mean(dim=(2, 3))                      # Eqn. 2: squeeze (B,C,H,W) -> (B,C)
        s = torch.relu(self.fc1(z))                # excite: FC1 + ReLU
        s = torch.sigmoid(self.fc2(s))             # Eqn. 3: FC2 + sigmoid -> gates in (0,1)
        return x * s.view(B, C, 1, 1)              # Eqn. 4: scale, broadcast over H,W


# --- 2. A tiny CNN; use_se toggles the SE blocks for the ablation. ---
class TinyCNN(nn.Module):
    def __init__(self, use_se=True, n_classes=10):
        super().__init__()
        self.use_se = use_se
        self.conv1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1, bias=False),
                                   nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.se1   = SEBlock(32)
        self.conv2 = nn.Sequential(nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
                                   nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.se2   = SEBlock(64)
        self.head  = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.conv1(x)
        if self.use_se: x = self.se1(x)            # recalibrate channels (ABLATION switch)
        x = self.conv2(x)
        if self.use_se: x = self.se2(x)
        x = x.mean(dim=(2, 3))                      # global average pool
        return self.head(x)


# --- 3. A CIFAR-10 subset (torchvision is preinstalled in Colab). ---
tfm = T.Compose([T.ToTensor(),
                 T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])
train_full = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=tfm)
test_full  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=tfm)
train_set  = torch.utils.data.Subset(train_full, range(5000))
test_set   = torch.utils.data.Subset(test_full,  range(2000))
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256)
device = "cuda" if torch.cuda.is_available() else "cpu"


def run(use_se, epochs=5):
    torch.manual_seed(0)
    net = TinyCNN(use_se=use_se).to(device)
    opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    lf  = nn.CrossEntropyLoss()
    for ep in range(epochs):
        net.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); lf(net(xb), yb).backward(); opt.step()
    net.eval(); correct = tot = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (net(xb).argmax(1) == yb).sum().item(); tot += yb.size(0)
    n_params = sum(p.numel() for p in net.parameters())
    return correct / tot, n_params

acc_se,    p_se    = run(use_se=True)
acc_plain, p_plain = run(use_se=False)
print(f"with SE   : val acc {acc_se:.3f}   params {p_se}")
print(f"no SE     : val acc {acc_plain:.3f}   params {p_plain}")
print(f"SE adds {p_se - p_plain} params (the two FC layers per block).")
# The SE net matches or edges ahead for a small extra param count.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does adding Squeeze-and-Excitation blocks to a tiny CNN raise validation accuracy, and at what parameter cost?_

In [ ]:
import torch, torch.nn as nn, torchvision, torchvision.transforms as T

# Reproduces the qualitative effect on a CIFAR-10 subset: SE blocks raise val
# accuracy for a tiny param cost, and the learned gates spread across (0,1).
class SEBlock(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(1, c // r)
        self.fc1, self.fc2 = nn.Linear(c, h), nn.Linear(h, c)
    def forward(self, x):
        B, C, _, _ = x.shape
        z = x.mean(dim=(2, 3))
        s = torch.sigmoid(self.fc2(torch.relu(self.fc1(z))))
        return x * s.view(B, C, 1, 1), s            # also return gates for the viz

class TinyCNN(nn.Module):
    def __init__(self, use_se):
        super().__init__()
        self.use_se = use_se
        self.c1 = nn.Sequential(nn.Conv2d(3,32,3,padding=1,bias=False), nn.BatchNorm2d(32), nn.ReLU())
        self.se1 = SEBlock(32)
        self.c2 = nn.Sequential(nn.Conv2d(32,64,3,stride=2,padding=1,bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.se2 = SEBlock(64)
        self.head = nn.Linear(64, 10)
        self.last_gates = None
    def forward(self, x):
        x = self.c1(x)
        if self.use_se: x, self.last_gates = self.se1(x)
        x = self.c2(x)
        if self.use_se: x, _ = self.se2(x)
        return self.head(x.mean(dim=(2,3)))

tfm = T.Compose([T.ToTensor(), T.Normalize((0.4914,0.4822,0.4465),(0.247,0.243,0.261))])
tr = torch.utils.data.Subset(torchvision.datasets.CIFAR10("./data",True,download=True,transform=tfm), range(5000))
te = torch.utils.data.Subset(torchvision.datasets.CIFAR10("./data",False,download=True,transform=tfm), range(2000))
trl = torch.utils.data.DataLoader(tr, batch_size=128, shuffle=True)
tel = torch.utils.data.DataLoader(te, batch_size=256)

def run(use_se):
    torch.manual_seed(0)
    net = TinyCNN(use_se); opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    lf = nn.CrossEntropyLoss()
    for _ in range(5):
        net.train()
        for xb, yb in trl:
            opt.zero_grad(); lf(net(xb), yb).backward(); opt.step()
    net.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in tel:
            c += (net(xb).argmax(1)==yb).sum().item(); t += yb.size(0)
    gates = net.last_gates[0].detach().sort().values.tolist() if use_se else None
    return c/t, sum(p.numel() for p in net.parameters()), gates

acc0, p0, _     = run(False)
acc1, p1, gates = run(True)
print("no SE  : acc", round(acc0,3), "params", p0)
print("with SE: acc", round(acc1,3), "params", p1, "  (+", p1-p0, ")")
print("sorted stage-1 gates:", [round(g,2) for g in gates])
# SE -> higher val acc for ~hundreds of extra params; gates spread across (0,1).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working tiny CNN with one SE block after its conv stage. Remove the
            SE block (so each channel passes through ungated) and retrain, everything else identical. What do
            you expect to happen to validation accuracy, and what does the comparison demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete only the SE block: keep the same convs, BatchNorm, ReLU, depth, optimizer, and data; just skip the x = se(x) call. — _An honest ablation changes exactly one thing &mdash; the channel recalibration &mdash; so any difference is attributable to it._
- Retrain and compare final validation accuracy of the SE net vs the plain net. — _The SE block's gates let useful channels speak up and suppress distracting ones, which should match or improve generalization at a tiny parameter cost._
- Note that the SE block can always learn gates $\approx 1$ (a no-op via the sigmoid), so it should rarely hurt. — _That safety is why the paper bolts SE onto existing nets &mdash; in the worst case it recovers the original features._

**Answer:** The SE version should reach equal-or-higher validation accuracy than the matched
                 plain net at a small extra parameter cost. Since the two nets differ only by the SE block,
                 this isolates channel recalibration as the cause. Because the sigmoid gates can sit near
                 $1$ (a no-op), the block can fall back to a near-identity, so it is safe to add. The CODEVIZ
                 panel shows this with/without contrast on a toy run.

</details>

**Problem 2.** A conv stage outputs $C = 64$ channels, each $8\times8$. You write an SE block with reduction
            $r = 16$. How many hidden units does the bottleneck have, and what are the shapes of $W_1$ and
            $W_2$? Roughly how many extra parameters does the block add (count the two FC weight matrices)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bottleneck width $= C/r = 64/16 = 4$ hidden units. — _The first FC layer squeezes the $C$ channel summaries down to $C/r$ (Eqn. 3)._
- $W_1$ is $\tfrac{C}{r}\times C = 4\times 64$; $W_2$ is $C\times\tfrac{C}{r} = 64\times 4$. — _$W_1$ maps $C\to C/r$, $W_2$ maps $C/r\to C$ &mdash; the down-then-up bottleneck._
- Params $\approx 4\cdot64 + 64\cdot4 = 256 + 256 = 512$ (plus small biases), i.e. $2C^2/r$. — _Far cheaper than a single $C\times C = 4096$-weight FC layer; the $1/r$ factor is the whole point of the bottleneck._

**Answer:** The bottleneck has $C/r = 4$ units. $W_1$ is $4\times64$ and $W_2$ is $64\times4$, adding
                 $\approx 4\cdot64 + 64\cdot4 = 512$ weights ($2C^2/r$) &mdash; tiny next to the convolutions.
                 That is why SE blocks improve accuracy "at slight additional computational cost."

</details>

**Problem 3.** In your worked example the gates came out $s = [0.5, 0.5]$ from toy weights. Suppose a trained
            block instead produced $s = [0.9, 0.1]$ for the same inputs
            $u_1 = \begin{bmatrix}4&0\\0&0\end{bmatrix}$, $u_2 = \begin{bmatrix}1&1\\1&1\end{bmatrix}$.
            What are the recalibrated channels, and what has the block decided?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply Eqn. 4 to channel 1: $\tilde{x}_1 = 0.9\,u_1 = \begin{bmatrix}3.6&0\\0&0\end{bmatrix}$. — _A gate near $1$ passes the channel through almost unchanged._
- Apply Eqn. 4 to channel 2: $\tilde{x}_2 = 0.1\,u_2 = \begin{bmatrix}0.1&0.1\\0.1&0.1\end{bmatrix}$. — _A gate near $0$ strongly suppresses the channel._
- Read off the decision: channel 1 (the bright-spot feature) is kept; channel 2 (the uniform feature) is turned down. — _The gates are the learned per-channel importance for this input &mdash; recalibration in action._

**Answer:** $\tilde{x}_1 = \begin{bmatrix}3.6&0\\0&0\end{bmatrix}$ (kept) and
                 $\tilde{x}_2 = \begin{bmatrix}0.1&0.1\\0.1&0.1\end{bmatrix}$ (suppressed). The block has
                 decided channel 1 matters here and channel 2 does not, scaling each channel's whole map by its
                 single gate &mdash; exactly the per-channel "volume control" the paper introduces.

</details>